In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
from detectron2.data.datasets.coco import load_coco_json
from pathlib import Path

base_path = Path.cwd()
coco_path = base_path / "data" / "coco" 
coco_annotations_path = coco_path / "annotations" 
coco_keypoints_path = coco_annotations_path / "person_keypoints_train2017.json"
coco_scalenet_results_path = coco_path / "coco_results" 
coco_images_root_path =  coco_path / "train2017"

coco_json = load_coco_json(coco_keypoints_path, coco_images_root_path)

In [ ]:
for x in coco_json:
    if x["annotations"]:
        print(x)
        break

In [ ]:
import pickle
import numpy as np
from scipy.io import loadmat
from detectron2.structures import BoxMode
import logging

class CocoScale2017:
    def __init__(
        self,
        split: str = "train",
        logger: logging.Logger | None = None,
        camera_parameters_file_path: str | Path = coco_scalenet_results_path / "yannick_results_train2017_filtered",
        coco_json_file_path: str | Path = coco_keypoints_path,
        coco_image_root_path: str | Path = coco_images_root_path,
        coco_scale_pickle_path: str | Path = coco_scalenet_results_path / "results_with_kps_20200208_morethan2_2-8" / "pickle", 
        debug: bool = False,
        shuffle: bool = False,
    ):
        if logger is None:
            self.logger = logging.getLogger(__name__)
        else:
            self.logger = logger

        self.coco_data = load_coco_json(coco_json_file_path, coco_image_root_path)
        self.is_train = split in ["train", "val"]

        # Estimated GT from coco files using a calibrated model?
        self.camera_parameters_files = sorted(Path(camera_parameters_file_path).glob("*.mat"), key=lambda x: str(x))
        random.shuffle(self.camera_parameters_files)
        if split == "train":
            self.camera_parameters_files = self.camera_parameters_files[: int(len(self.camera_parameters_files) * 0.8)]
            img_filenames = [
                str(camera_mat_file.name).split(".")[0]
                for camera_mat_file in self.camera_parameters_files
            ]

            self.img_files = [
                coco_image_root_path / (img_filename + ".jpg")
                for img_filename in img_filenames
            ] 

            self.pickle_files = [
                coco_scale_pickle_path / (img_filename + ".data")
                for img_filename in img_filenames
            ] 
        else:
            # No camera pre-information in val / test
            self.camera_parameters_files = []
            self.pickle_files = sorted(Path(coco_scale_pickle_path).glob("*.data"), key=lambda x: str(x))
            self.img_files = [
                coco_image_root_path / (str(pickle_file.stem()) + ".jpg")
                for pickle_file in self.pickle_files
            ] 
        if debug:
            self.pickle_files = self.pickle_files[:100]
            self.img_files = self.img_files[:100]

        assert len(self.img_files) == len(self.pickle_files)
        print(self.camera_parameters_files, self.pickle_files, self.img_files)
        if shuffle and self.is_train:
            list_zip = list(zip(self.img_files, self.pickle_files))
            random.shuffle(list_zip)
            self.img_files, self.pickle_files = zip(*list_zip)

    def __getitem__(self, k):
        with open(self.pickle_files[k], "rb") as fhdl:
            data = pickle.load(fhdl)
        im_path = self.img_files[k]
        bboxes = data["bboxes"].astype(np.float32)
        kps_gt = data["kps"].astype(int).tolist()
        horizon = -1
        vfov = -1
        if self.is_train:
            camera_parameters = loadmat(self.camera_parameters_files[k])
            horizon = camera_parameters["pitch"][0][0].astype(np.float32)
            vfov = camera_parameters["vfov"][0][0].astype(np.float32)
        instances = []
        for bbox, kps in zip(bboxes, kps_gt):
            instances.append(dict(
                bbox=bbox.tolist(),
                bbox_mode=BoxMode.XYWH_ABS,
                category_id=0,
                keypoints=kps,
            ))
        return dict(
            file_name=im_path,
            image_id=im_path,
            horizon=horizon,
            vfov=vfov,
            annotations=instances
        )

    def get_all_items(self):
        for i, _ in enumerate(self.pickle_files):
            yield self[i]

    def __call__(self):
        return self

    def __len__(self):
        return len(self.pickle_files)

In [ ]:
debug = True
coco_scale_train = CocoScale2017(debug=debug)

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_dataset_name = "CocoScale2017_train"
DatasetCatalog.register(coco_scale_dataset_name, coco_scale_train)
MetadataCatalog.get(coco_scale_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

if debug and len(coco_scale_train) < 500:
    dataset_dicts = list(coco_scale_train.get_all_items())
    for i, d in enumerate(random.sample(dataset_dicts, 3)):
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5, metadata=MetadataCatalog.get(coco_scale_dataset_name))
        visualizer.draw_dataset_dict(d)
        out = visualizer.get_output()
        img = out.get_image()
        plt.imshow(img)
        plt.show()

In [ ]:
import os
from detectron2.engine import DefaultTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (coco_scale_dataset_name, )
cfg.DATASETS.TEST = (coco_scale_dataset_name, )
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 4  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.MODEL.KEYPOINT_ON=True
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=True
cfg.SOLVER.MAX_ITER = 500
experiment_name = "test-debug-kpsbbox-dataset"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.

trainer = DefaultTrainer(cfg) 
# NOTE: change value of resume if we have a last_checkpoint
trainer.resume_or_load(resume=False)

In [ ]:
!export PATH=/opt/cuda/bin${PATH:+:${PATH}}
!export LD_LIBRARY_PATH=/opt/cuda/lib64
!nvcc --version

In [ ]:
trainer.train()